# 04 · Representation Learning (Autoencoder)

**Goal (PDF 3.5 + 3.6):**
1. Train autoencoder on TF-IDF vectors
2. Use latent representations for classification
3. Compare with raw TF-IDF results from notebook 03
4. Intra-class vs inter-class cosine similarity
5. t-SNE visualization

**MLflow tag:** `study=autoencoder`

> Works both locally and on Google Colab. Set `RETRAIN=False` to skip training and load saved encoder.

## 0. Environment setup

In [2]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/Computers/My laptop/imdb-sentiment-classification'
    import subprocess
    subprocess.run(['pip', 'install', 'mlflow', '-q'])
else:
    try:
        torch_lib = os.path.join(
            os.environ.get('VIRTUAL_ENV', ''),
            'Lib', 'site-packages', 'torch', 'lib'
        )
        if os.path.exists(torch_lib):
            os.add_dll_directory(torch_lib)
    except Exception:
        pass
    BASE_PATH = os.path.abspath('..')

RESULTS_PATH = f'{BASE_PATH}/results'
sys.path.insert(0, BASE_PATH)

import torch
print(f'Torch:  {torch.__version__}')
print(f'Colab:  {IN_COLAB}')
print(f'GPU:    {torch.cuda.is_available()}')
print(f'Base:   {BASE_PATH}')

Mounted at /content/drive
Torch:  2.10.0+cpu
Colab:  True
GPU:    False
Base:   /content/drive/Computers/My laptop/imdb-sentiment-classification


In [3]:
import os

# Pokušaj različite putanje
paths = [
    '/content/drive/MyDrive',
    '/content/drive/My Drive',
    '/content/drive/Computers',
]

for p in paths:
    if os.path.exists(p):
        print(f'EXISTS: {p}')
        print(os.listdir(p))
    else:
        print(f'NOT FOUND: {p}')

EXISTS: /content/drive/MyDrive
['IMG_20180201_070142 (1).jpg', 'IMG_20180503_193304.jpg', 'IMG_20180614_165913.jpg', 'IMG_20180112_110853.jpg', 'IMG_20180112_093020.jpg', 'IMG_20180516_200339.jpg', 'IMG_20180611_174918.jpg', 'IMG_20180528_192939.jpg', 'IMG_20180619_135512.jpg', 'IMG_20180503_190107.jpg', 'IMG_20180503_190045.jpg', 'IMG_20180619_135537.jpg', 'IMG_20171116_181548.jpg', 'IMG_20180112_111724.jpg', 'IMG_20180424_193739.jpg', 'IMG_20180528_191614.jpg', 'IMG_20180528_192158.jpg', 'IMG_20180328_112410.jpg', 'IMG_20180503_190711.jpg', 'IMG_20180619_103400.jpg', 'IMG_20180503_185921.jpg', 'IMG_20180530_184527.jpg', 'IMG_20180424_183643.jpg', 'IMG_20180619_135557.jpg', 'IMG_20180220_132857.jpg', 'IMG_20180503_190706.jpg', 'IMG_20171113_163817.jpg', 'IMG_20180420_155704.jpg', 'IMG_20180528_191632.jpg', 'IMG_20180112_111540.jpg', 'IMG_20180619_135552.jpg', 'IMG_20180529_165949.jpg', 'IMG_20180112_093816.jpg', 'IMG_20180112_095424.jpg', 'IMG_20180423_193025.jpg', 'IMG_20180112_09232

In [4]:
import pickle
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE

from models.autoencoder import TfidfAutoencoder
from models.svm import SVMModel
from evaluation.metrics import ClassificationMetrics

ModuleNotFoundError: No module named 'models'

## 1. Load data and vectorize

Using the same vectorizer as notebook 03 — identical feature space for fair comparison.

In [ ]:
with open(f'{RESULTS_PATH}/splits.pkl', 'rb') as f:
    splits = pickle.load(f)

df_train = splits['train']
df_val   = splits['val']
df_test  = splits['test']

with open(f'{RESULTS_PATH}/shared_vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

X_train = vectorizer.transform(df_train)
X_val   = vectorizer.transform(df_val)
X_test  = vectorizer.transform(df_test)

y_train = df_train['sentiment'].values
y_val   = df_val['sentiment'].values
y_test  = df_test['sentiment'].values

print(f'X_train: {X_train.shape}')
print(f'X_val:   {X_val.shape}')
print(f'X_test:  {X_test.shape}')

## 2. MLflow setup

In [ ]:
if IN_COLAB:
    mlflow.set_tracking_uri(f'file://{BASE_PATH}/mlruns')
else:
    mlflow.set_tracking_uri(f'file:///{BASE_PATH}/mlruns'.replace('\\', '/'))

mlflow.set_experiment('imdb-sentiment')

## 3. Train autoencoder

Architecture: `input_dim → 1024 → 256 → 1024 → input_dim`

- **Loss:** MSE — measures reconstruction quality
- **Optimizer:** Adam with lr=1e-3
- **Labels not used** — unsupervised, learns from reconstruction only

Set `RETRAIN = False` to skip training and load a previously saved encoder.

In [ ]:
RETRAIN = True

input_dim  = X_train.shape[1]
hidden_dim = 1024
latent_dim = 256
n_epochs   = 20
batch_size = 64
lr         = 1e-3

autoencoder = TfidfAutoencoder(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    latent_dim=latent_dim,
    n_epochs=n_epochs,
    batch_size=batch_size,
    learning_rate=lr,
)

ENCODER_PATH = f'{RESULTS_PATH}/encoder.pt'

if RETRAIN:
    with mlflow.start_run(run_name='autoencoder_training', tags={'study': 'autoencoder'}):
        mlflow.log_params(autoencoder.get_params())

        history = autoencoder.train(X_train, X_val)

        for epoch, (tl, vl) in enumerate(zip(history['train_loss'], history['val_loss'])):
            mlflow.log_metrics({'train_loss': tl, 'val_loss': vl}, step=epoch)

        mlflow.log_metric('training_time_s', autoencoder.training_time)

    torch.save(autoencoder._encoder.state_dict(), ENCODER_PATH)
    print(f'Encoder saved. Training time: {autoencoder.training_time:.1f}s')
else:
    autoencoder._encoder.load_state_dict(
        torch.load(ENCODER_PATH, map_location=autoencoder.device)
    )
    autoencoder._is_trained = True
    print(f'Encoder loaded from {ENCODER_PATH}')

## 4. Reconstruction loss curve

Train and val loss should both decrease and converge together.  
If val loss rises while train loss falls — overfitting.

In [ ]:
if RETRAIN:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(history['train_loss'], label='Train loss')
    ax.plot(history['val_loss'],   label='Val loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('Autoencoder Reconstruction Loss')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('Loss curve not available — encoder loaded, not trained.')

## 5. Extract latent representations

Pass all splits through the encoder only.  
Output shape: `(n_samples, 256)` — 547x compression from 139992 features.

In [ ]:
X_train_latent = autoencoder.encode(X_train)
X_val_latent   = autoencoder.encode(X_val)
X_test_latent  = autoencoder.encode(X_test)

print(f'Latent train: {X_train_latent.shape}')
print(f'Latent val:   {X_val_latent.shape}')
print(f'Latent test:  {X_test_latent.shape}')
print(f'Compression:  {input_dim} → {latent_dim} ({input_dim // latent_dim}x)')

## 6. Classification on latent representations

Same SVM as notebook 03 — only the feature space changes (256 vs 139992).

In [ ]:
svm_latent = SVMModel()

t0 = time.perf_counter()
svm_latent.train(X_train_latent, y_train)
train_time = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_test = svm_latent.predict(X_test_latent)
inf_time = time.perf_counter() - t0

latent_metrics = ClassificationMetrics(
    y_true=y_test,
    y_pred=y_pred_test,
    training_time=train_time,
    inference_time=inf_time,
    n_features=X_test_latent.shape[1],
)

with mlflow.start_run(run_name='svm_on_latent', tags={'study': 'autoencoder'}):
    mlflow.log_params({
        'model':      'SVMModel',
        'features':   'autoencoder_latent',
        'latent_dim': latent_dim,
    })
    mlflow.log_metrics({f'test_{k}': v for k, v in latent_metrics.to_dict().items()})

print(latent_metrics.report())

## 7. Comparison: raw TF-IDF vs latent representations

Update `RAW_TFIDF_F1` and `RAW_TFIDF_ACCURACY` with your actual values from notebook 03.

In [ ]:
RAW_TFIDF_F1       = 0.9073  # from notebook 03 — SVM best config
RAW_TFIDF_ACCURACY = 0.9072

comparison = pd.DataFrame([
    {
        'features':        'Raw TF-IDF',
        'n_features':      input_dim,
        'test_f1':         RAW_TFIDF_F1,
        'test_accuracy':   RAW_TFIDF_ACCURACY,
    },
    {
        'features':        f'Autoencoder latent (dim={latent_dim})',
        'n_features':      latent_dim,
        'test_f1':         round(latent_metrics.f1, 4),
        'test_accuracy':   round(latent_metrics.accuracy, 4),
    },
])
comparison['compression_ratio'] = (input_dim / comparison['n_features']).round(0).astype(int).astype(str) + 'x'
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(comparison['features'], comparison['test_f1'],
              color=['steelblue', 'darkorange'])
ax.bar_label(bars, fmt='%.4f', padding=3)
ax.set_ylim(0.8, 1.0)
ax.set_ylabel('Test F1')
ax.set_title('Raw TF-IDF vs Autoencoder Latent Representations')
plt.tight_layout()
plt.show()

## 8. Representation Analysis (PDF 3.6)

### 8.1 Intra-class vs inter-class cosine similarity

If intra-class similarity > inter-class — latent space groups similar reviews together.

In [ ]:
N_SAMPLE = 1000
np.random.seed(42)
idx = np.random.choice(len(X_test_latent), N_SAMPLE, replace=False)

X_sample = X_test_latent[idx]
y_sample = y_test[idx]

X_pos = X_sample[y_sample == 1]
X_neg = X_sample[y_sample == 0]

sim_pos = cosine_similarity(X_pos)
sim_neg = cosine_similarity(X_neg)
np.fill_diagonal(sim_pos, np.nan)
np.fill_diagonal(sim_neg, np.nan)

intra_pos = np.nanmean(sim_pos)
intra_neg = np.nanmean(sim_neg)
inter     = np.mean(cosine_similarity(X_pos, X_neg))

print(f'Intra-class similarity (positive): {intra_pos:.4f}')
print(f'Intra-class similarity (negative): {intra_neg:.4f}')
print(f'Inter-class similarity:            {inter:.4f}')
print(f'\nIntra > Inter: {(intra_pos + intra_neg) / 2 > inter}')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ['Intra (positive)', 'Intra (negative)', 'Inter'],
    [intra_pos, intra_neg, inter],
    color=['steelblue', 'steelblue', 'tomato']
)
ax.bar_label(bars, fmt='%.4f', padding=3)
ax.set_ylabel('Mean Cosine Similarity')
ax.set_title('Intra-class vs Inter-class Cosine Similarity\n(Latent Representations)')
plt.tight_layout()
plt.show()

### 8.2 t-SNE visualization

Projects 256-dimensional latent space to 2D.  
Separated clusters = autoencoder learned sentiment-relevant representations.

In [ ]:
print('Running t-SNE (this may take a minute)...')

tsne   = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_sample)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['tomato' if y == 0 else 'steelblue' for y in y_sample]
ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=colors, alpha=0.5, s=10)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='tomato',    label='Negative'),
    Patch(color='steelblue', label='Positive'),
])
ax.set_title('t-SNE of Autoencoder Latent Representations')
ax.set_xlabel('t-SNE dim 1')
ax.set_ylabel('t-SNE dim 2')
plt.tight_layout()
plt.show()

### 8.3 Confusion matrix — SVM on latent representations

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    latent_metrics.confusion_matrix,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=['negative', 'positive'],
    yticklabels=['negative', 'positive'],
    ax=ax,
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — SVM on Latent (F1={latent_metrics.f1:.4f})')
plt.tight_layout()
plt.show()